In [ ]:
# LAst amended: 26th April, 2026
# Ref: https://developers.llamaindex.ai/python/framework/integrations/vector_stores/elasticsearchindexdemo/

<hr style="height:10px; border:none; color:#333; background-color:grey;" />


::Q:: Which all vector stores provide: a) Dense vector Search; b) Keyword search, c) Hybrid search and d) Metadata filtering

**::Ans::**
>According to the official documentation, vector stores that support all of: (a) dense vector search, (b) keyword search, (c) hybrid search, and (d) metadata filtering include Elasticsearch, Milvus (with full-text/hybrid search), Qdrant, Pinecone (hybrid/metadata), Weaviate, and Google Vertex AI Vector Search v2.0. These features are listed in the integration and advanced retrieval guides

**::Q::** Did you miss PGVector store and Elasticsearch store


**::Ans::**
>No, Elasticsearch store was included in the previous answer. PGVector (Postgres with pgvector extension) also supports dense vector search, keyword search (via Postgres full-text), hybrid search, and metadata filtering (LlamaCloud Retrieval Advanced, PGVectorStore source).

<hr style="height:10px; border:none; color:#333; background-color:grey;" />


# Elasticsearch Vector Store

>Get started with Elasticsearch by running it locally. The easiest way is to use the official Elasticsearch Docker image. See the Elasticsearch Docker documentation for more information.

Install Elasticsearch
<pre>
docker run -p 9200:9200 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  -e "xpack.license.self_generated.type=trial" \
  docker.elastic.co/elasticsearch/elasticsearch:8.13.2
</pre>    

>This installation gives a unique name to ElasticSearch. Next time use that name.

## Configuring ElasticsearchStore     
The ElasticsearchStore class is used to connect to an Elasticsearch instance. It requires the following parameters:

<pre>
    - index_name: Name of the Elasticsearch index. Required.
    - es_client: Optional. Pre-existing Elasticsearch client.
    - es_url: Optional. Elasticsearch URL.
    - es_cloud_id: Optional. Elasticsearch cloud ID.
    - es_api_key: Optional. Elasticsearch API key.
    - es_user: Optional. Elasticsearch username.
    - es_password: Optional. Elasticsearch password.
    - text_field: Optional. Name of the Elasticsearch field that stores the text.
    - vector_field: Optional. Name of the Elasticsearch field that stores the
                embedding.
    - batch_size: Optional. Batch size for bulk indexing. Defaults to 200.
    - distance_strategy: Optional. Distance strategy to use for similarity search.
                    Defaults to "COSINE".
    
</pre>

In [1]:
# Install relevant library
%pip install -qU llama-index-vector-stores-elasticsearch 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Using Ollama

In [14]:
# 1.0 Ollama related
#     https://docs.llamaindex.ai/en/stable/examples/embeddings/ollama_embedding/
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings

In [15]:
# 1.1

llm = Ollama(
                 model=  "deepseek-r1:7b", #  works best for router agent, # "qwen3.5:4b", #      "llama3.2:latest",
                 request_timeout=3600,  # Change it
                 #temperature = 0.4,
                 context_window=4096,    # Limit timeouts
                 #mirostat = 0
            )

Settings.llm = llm

embed_model = OllamaEmbedding(
                              model_name="nomic-embed-text",      
                              base_url="http://localhost:11434",
                             )

Settings.embed_model = embed_model

### Connecting locally

In [16]:
# 2.0 Ihis step is redundant, as seen as below
#   We create another index below:

from llama_index.vector_stores.elasticsearch import ElasticsearchStore

# 2.0.1
es = ElasticsearchStore(
                        index_name="my_index",
                        es_url="http://localhost:9200",
                        )

## Example data

In [17]:
# 3.0
from llama_index.core.schema import TextNode

movies = [
    TextNode(
        text="The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.",
        metadata={"title": "Pulp Fiction"},
    ),
    TextNode(
        text="When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.",
        metadata={"title": "The Dark Knight"},
    ),
    TextNode(
        text="An insomniac office worker and a devil-may-care soapmaker form an underground fight club that evolves into something much, much more.",
        metadata={"title": "Fight Club"},
    ),
    TextNode(
        text="A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.",
        metadata={"title": "Inception"},
    ),
    TextNode(
        text="A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its controllers.",
        metadata={"title": "The Matrix"},
    ),
    TextNode(
        text="Two detectives, a rookie and a veteran, hunt a serial killer who uses the seven deadly sins as his motives.",
        metadata={"title": "Se7en"},
    ),
    TextNode(
        text="An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.",
        metadata={"title": "The Godfather", "theme": "Mafia"},
    ),
]

<hr style="height:10px; border:none; color:#333; background-color:grey;" />


**::Q::**
>When I apply metadata filter then is the whole index searched even then or the scope of index search would be reduced thereby making search faster

>When you apply a metadata filter, the scope of the index search is reduced: only nodes/documents matching the filter are considered for similarity search, which can make the search faster and more efficient. This is supported for vector stores that natively support metadata filtering, such as PGVectorStore and ElasticsearchStore, as shown in the LlamaIndex metadata filtering documentation.

### More on reading data      
Reading text data from files and at the same time creating metadata

**::Q::**   
> In the example for the ElasticSearch vector store, I have several TextNodes taking the inputs. TextNode takes two inputs, text and metadata. Can I have a code something like this: movies=[ TextNode( text = SomeFileReader(Textfilename1), metadata={"title" : "case number xyz1"} ), TextNode( text = SomeFileReader(Textfilename2), metadata={"title" : "case number xyz2"} ), ] If yes, please suggest a script where I read multiple files, and for each I add metadata separately. Also can TextNode take .md files as input

**Ans**:
>Yes, you can create a list of TextNodes by reading multiple files and assigning separate metadata to each, as shown in your example. You simply need to read the file content (regardless of extension, including .md) and pass it as the text argument to TextNode, with metadata as a dictionary. TextNode does not care about file type; it just takes a string for text, so .md files are fine if you read them as text.

Here are examples

In [ ]:
from llama_index.core.schema import TextNode

filenames = ["file1.md", "file2.md"]
movies = []
for fname in filenames:
    with open(fname, "r", encoding="utf-8") as f:
        content = f.read()
    movies.append(TextNode(text=content, metadata=("title": f"case number {fname)"}))


**::Q::**
>Thanks. Please change the above example code where all files are read automatically from a folder without requiring their prior listing .

In [ ]:
import os
from llama_index.core.schema import TextNode

folder = "./your_folder"
movies = []
for fname in os.listdir(folder):
    path = os.path.join(folder, fname)
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        movies.append(TextNode(text=content, metadata=("title": f"case number {fname)"}))


**::Q::**
>Ok. Let me put it differently. In the above example, a file is read from a folder and its title is taken as the metadata for the TextNode. Now all my files are several sub-folders (and not directly in the parent folder). Each sub-folder has a meaningful name. My code should read all text files from each one of the sub-folders, one-by-one, and now text of each file read has two metadata points: one is the filename as 'title' and the other is the sub-folder name as 'region'. What would be code that creates several TextNodes from each file read and appends them to movies = []

In [ ]:
import os
from llama_index.core.schema import TextNode

parent_folder = "./your_parent_folder"
movies = []
for root, dirs, files in os.walk(parent_folder):
    subfolder = os.path.basename(root)
    for fname in files:
        path = os.path.join(root, fname)
        if os.path.isfile(path):
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
            movies.append(TextNode(
                text=content,
                metadata=("title": fname, "region": subfolder)
            ))


>This will create a TextNode for every file in every subfolder, with title as the filename and region as the subfolder name,

**::Q::**
Can TextNode be also used when the storage_context is PGVector and not elasticsearch. If yes what would be the code?

>Yes, you can use TextNode with PGVectorStore as your vector store backend. TextNode is a general-purpose node type in LlamaIndex and works with any supported vector store, including PGVector. The process is the same: create TextNodes, then build a VectorStoreIndex with a StorageContext that uses PGVectorStore.

In [ ]:
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.schema import TextNode

# Assume movies is a list of TextNode objects as before
# Define a PGVector store
vector_store = PGVectorStore.from_params(
                                        database="vector_db",
                                        host="localhost",
                                        password="password",
                                        port=5432,
                                        user="postgres",
                                        table_name="my_table",
                                        embed_dim=1536  # embedding dimension
                                    )

# USe the above PGVector store as our storage context
storage_context = StorageContext.from_defaults(vector_store=vector_store)
# Fill this storage context with indicies
index = VectorStoreIndex(movies, storage_context=storage_context)


**::Q::** When I apply metadata filter then is the whole index searched even then or the scope of index search would be reduced thereby making search faster

>When you apply a metadata filter, the scope of the index search is reduced: only nodes/documents matching the filter are considered for similarity search, which can make the search faster and more efficient. This is supported for vector stores that natively support metadata filtering, such as PGVectorStore and ElasticsearchStore, as shown in the LlamaIndex metadata filtering documentation.

<hr style="height:10px; border:none; color:#333; background-color:grey;" />


In [18]:
# 4.0
from llama_index.core import StorageContext, VectorStoreIndex

# 4.0.1 Importing ElasticsearchStore  again!
from llama_index.vector_stores.elasticsearch import ElasticsearchStore

### Define functions for retrieval results

In [23]:
# 5.0 This function is OK

def print_results(results):
    for rank, result in enumerate(results, 1):
        print(
            f"{rank}. title={result.metadata['title']} score={result.get_score()} text={result.get_text()}"
        )

In [20]:
# 5.1 Can do without it:

def search( vector_store: ElasticsearchStore, nodes: list[TextNode], query: str):
    # Create storage context
    storage_context = StorageContext.from_defaults(
                                                   vector_store=vector_store
                                                  )
    # Create index
    index = VectorStoreIndex(nodes, storage_context=storage_context)

    print(">>> Documents:")
    retriever = index.as_retriever()
    results = retriever.retrieve(query)
    print_results(results)

    print("\n>>> Answer:")
    query_engine = index.as_query_engine()
    response = query_engine.query(query)
    print(response)

In [43]:
# 5.1 Dense Vector Stragetgy is default
from llama_index.vector_stores.elasticsearch import AsyncDenseVectorStrategy

# 5.2 Connect to Elasticsearch store again
dense_vector_store = ElasticsearchStore(
                                        es_url="http://localhost:9200",  # for Elastic Cloud authentication see above
                                        index_name="movies_metadata",
                                        retrieval_strategy=AsyncDenseVectorStrategy(),   # This is default
                                        )

In [44]:
# 6.0
storage_context = StorageContext.from_defaults(
                                                vector_store= dense_vector_store
                                              )

# 6.1
index = VectorStoreIndex(
                          movies, 
                          storage_context=storage_context
                        )


In [46]:
# 6.2
query = "which movie involves dreaming?"

In [77]:
# 6.3 Retrieve relevant chunks:

retriever = index.as_retriever()
results = retriever.retrieve(query)
print_results(results)

1. title=Inception score=1.0 text=A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.
2. title=Fight Club score=0.0 text=An insomniac office worker and a devil-may-care soapmaker form an underground fight club that evolves into something much, much more.


In [48]:
# 6.4 Get query engine:

print("\n>>> Answer:")
query_engine = index.as_query_engine()
response = query_engine.query(query)
print(response)


>>> Answer:
The movie that involves the concept of a dream is "Inception."


## Keyword retrieval

>**AsyncBM25Strategy()** enables keyword-based retrieval using the BM25 algorithm, which ranks documents by exact term matches and their frequency. **AsyncDenseVectorStrategy()** enables dense vector retrieval, using embeddings to find semantically similar documents based on vector similarity (e.g., cosine distance).

>BM25 is best for exact keyword matches, while dense vector strategies excel at semantic search where meaning matters more than exact words (ElasticsearchIndexDemo, ElasticsearchStore API).

In [55]:
# 7.0
from llama_index.vector_stores.elasticsearch import AsyncBM25Strategy

bm25_store = ElasticsearchStore(
                                es_url="http://localhost:9200",  # for Elastic Cloud authentication see above
                                index_name="movies_bm25",
                                retrieval_strategy = AsyncBM25Strategy(),
                                )



In [56]:
# 7.1
storage_context = StorageContext.from_defaults(
                                                vector_store= bm25_store
                                              )

# 7.2
index = VectorStoreIndex(
                          movies, 
                          storage_context=storage_context
                        )


In [57]:
# 7.3
query = "joker"

In [58]:
# 7.4
retriever = index.as_retriever()
results = retriever.retrieve(query)
print_results(results)

1. title=The Dark Knight score=1.0 text=When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.


In [59]:
# 8.0
print("\n>>> Answer:")
query_engine = index.as_query_engine()
response = query_engine.query(query)
print(response)


>>> Answer:
The Joker is known for causing chaos and rejoicing in the suffering of his victims.


<hr style="height:10px; border:none; color:#333; background-color:grey;" />


**::Q::**
> My TextNode is as below. The word 'joker' exists in the text. My MetadataFilter has no keyword value as 'joker'. How come then I am applying AsyncBM25Strategy() strategy to answer the 'query = "joker"'.
> 
<pre>
TextNode( 
    text="When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.", 
    metadata={"title": "The Dark Knight"}
        )
</pre>    

**::Ans::**
>If your MetadataFilter does not filter on the keyword 'joker', but you use AsyncBM25Strategy() and query "joker", the retriever will still return the TextNode because BM25 matches based on the presence of the query term in the text, not in the metadata. MetadataFilters only restrict results if you explicitly filter on metadata fields; <u>otherwise, BM25 operates on the text content</u> 

**::Q::**
>What does *retriever = index.as_retriever(filters=filters)* do?

**::Ans::**
>The call *retriever = index.as_retriever(filters=filters)* creates a retriever object from your index that will only return nodes (document chunks) matching the specified metadata filters during retrieval. This means any subsequent *queries* using this retriever will be restricted to results that satisfy the filter conditions you provided, such as filtering by author, date, category, or other metadata fields.

<hr style="height:10px; border:none; color:#333; background-color:grey;" />


## Hybrid retrieval
Does not work. Fails

Combining dense retrieval and keyword search for hybrid retrieval can be enabled by setting a flag.

In [67]:
from llama_index.vector_stores.elasticsearch import AsyncDenseVectorStrategy

hybrid_store = ElasticsearchStore(
                                    es_url="http://localhost:9200",  # for Elastic Cloud authentication see above
                                    index_name="movies_hybrid",
                                    retrieval_strategy=AsyncDenseVectorStrategy(hybrid=True),
                                )


In [68]:
storage_context = StorageContext.from_defaults(
                                                vector_store= hybrid_store
                                              )

index = VectorStoreIndex(
                          movies, 
                          storage_context=storage_context
                        )


In [69]:
query = "which movie involves dreaming?"

In [ ]:
print("\n>>> Answer:")
query_engine = index.as_query_engine()
response = query_engine.query(query)
print(response)

## Metadata Filters
Good strategy

We can also apply filters to the query engine based on the metadata of our documents.

In [72]:
# 9.0
from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters
from llama_index.core.query_engine import RetrieverQueryEngine

# Create a metadata_store
#   The index_name 'movies_metadata' is used internally by ElasticsearchStore 
#     to specify which Elasticsearch index to store and retrieve your data from. 
#       When you create or query data using this store, all operations are directed 
#        to the 'movies_metadata' index in your Elasticsearch instance.
#   You use this name by referencing it when creating the ElasticsearchStore, 
#     as in this example. If you want to access, update, or query this data later,
#        you would again instantiate ElasticsearchStore with index_name="movies_metadata" 
#         to connect to the same index (ElasticsearchIndexDemo).

# 9.1
metadata_store = ElasticsearchStore(
                                    es_url=     "http://localhost:9200",  # for Elastic Cloud authentication see above
                                    index_name= "movies_metadata",
                                   )

# 9.2
storage_context = StorageContext.from_defaults(
                                                vector_store= metadata_store
                                              )

# 9.3
index = VectorStoreIndex(
                          movies, 
                          storage_context=storage_context
                        )

In [73]:
# 10.0 Write your Metadata filter:

filters = MetadataFilters(
                            filters=[ExactMatchFilter(key="theme", value="Mafia")]
                         )

In [74]:
# 10.1 Get pointer to nodes (chunks) matching
#       the specified metadata filter:

my_retriever = index.as_retriever(filters=filters)

In [75]:
# 11.0 Use that pointer to return nodes matching the query:
results = my_retriever.retrieve("What is inception about?")
print_results(results)

print( f"No of nodes returned: {len(results)}")

1. title=The Godfather score=1.0 text=An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.
No of nodes returned: 1


In [76]:
# 12.0 Create a query engine to retrieve results:

query_engine = RetrieverQueryEngine.from_args(my_retriever)
response = query_engine.query("What is inception about?")
print("=====")
print(response)

=====
The context provided discusses "The Godfather," which centers around a mafia dynasty's transfer of control from an aging patriarch to his son. However, it does not address or relate to "Inception." If you're asking about "Inception," please refer to its own separate narrative and information.


>To query or manage the *'movies_metadata'* index, you simply create a new ElasticsearchStore with the same index_name and use it to build a StorageContext and VectorStoreIndex. This allows you to retrieve, filter, or update data in that specific Elasticsearch index.

>For example, to query the index:

<pre>
metadata_store = ElasticsearchStore(
                                    es_url="http://localhost:9200",
                                    index_name="movies_metadata",
                                    )

storage_context = StorageContext.from_defaults(vector_store=metadata_store)
index = VectorStoreIndex.from_vector_store(metadata_store, storage_context=storage_context)

query_engine = index.as_query_engine()
response = query_engine.query("What is inception about?")
print(response)
</pre>

> This code connects to the existing 'movies_metadata' index and allows you to run queries or apply filters as needed

In [78]:
################